In [ ]:
import pandas as pd
import itertools
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets
from IPython.display import display
import traceback
from sklearn.metrics import mean_absolute_percentage_error

In [ ]:

def load_and_clean_data():
    merged_file = '/content/FINAL_MASTER_DATASET.csv'

    df = pd.DataFrame()

    try:
        df = pd.read_csv(merged_file)
        print(f"Success: Loaded {merged_file}")
    except FileNotFoundError:
        print(f"Note: '{merged_file}' not found. Attempting to merge raw files on the fly...")
        try:
            # Fallback to Option B (Merge on the fly)
            da = pd.read_csv(da_file)
            weather = pd.read_csv(weather_file)

            str_cols = ['MUNICIPALITY', 'FARM TYPE', 'MONTH', 'CROP']
            for col in str_cols:
                if col in da.columns: da[col] = da[col].astype(str).str.upper().str.strip()

            # Map Month
            month_map = {'JAN':1, 'FEB':2, 'MAR':3, 'APR':4, 'MAY':5, 'JUN':6,
                         'JUL':7, 'AUG':8, 'SEP':9, 'OCT':10, 'NOV':11, 'DEC':12}
            da['Month_Num'] = da['MONTH'].map(month_map)

            # Clean Weather
            if 'Date' in weather.columns:
                weather['Date'] = pd.to_datetime(weather['Date'])
                weather['YEAR'] = weather['Date'].dt.year
                weather['Month_Num'] = weather['Date'].dt.month

            weather_agg = weather.groupby(['YEAR', 'Month_Num'])['Rain Value (mm)'].sum().reset_index()

            # Merge
            df = pd.merge(da, weather_agg, on=['YEAR', 'Month_Num'], how='left')

        except FileNotFoundError:
            print("ERROR: Could not find dataset files. Please upload 'Benguet_Chronological_Dataset.csv' OR both raw files.")
            return pd.DataFrame()

    cols_to_fix = ['Production(mt)', 'Productivity(mt/ha)', 'Rain Value (mm)']
    for col in cols_to_fix:
        if col in df.columns:
            # If it's text, remove commas first
            if df[col].dtype == 'object':
                df[col] = df[col].astype(str).str.replace(',', '', regex=False)
            # Force convert to number (turn errors into NaN, then 0)
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # Standardize Text Columns
    text_cols = ['MUNICIPALITY', 'FARM TYPE', 'MONTH', 'CROP']
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.upper().str.strip()

    # Setup Date Column
    month_map = {'JAN':1, 'FEB':2, 'MAR':3, 'APR':4, 'MAY':5, 'JUN':6,
                 'JUL':7, 'AUG':8, 'SEP':9, 'OCT':10, 'NOV':11, 'DEC':12}

    if 'Date' not in df.columns:
        if 'Month_Num' not in df.columns:
             df['Month_Num'] = df['MONTH'].map(month_map)
        df['Date'] = pd.to_datetime(df['YEAR'].astype(str) + '-' + df['Month_Num'].astype(str) + '-01')
    else:
        df['Date'] = pd.to_datetime(df['Date'])

    return df.sort_values('Date')

df = load_and_clean_data()

def create_features(data, target_col, lags=[1, 2, 12]):
    df_feat = data.copy()
    for lag in lags:
        df_feat[f'lag_{lag}'] = df_feat[target_col].shift(lag)
    df_feat['Month'] = df_feat.index.month
    df_feat['Year'] = df_feat.index.year
    return df_feat.dropna()

def calculate_accuracy(data, target_col, feature_cols):
    if len(data) < 24: return "N/A (Need more data)"

    # Train on past, Test on last 12 months
    train_data = data.iloc[:-12]
    test_data = data.iloc[-12:]

    train_feat = create_features(train_data, target_col)
    X_train = train_feat[feature_cols]
    y_train = train_feat[target_col]

    rf_val = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_val.fit(X_train, y_train)

    # Test
    # Simplification: calculate features on full dataset then slice
    full_feat = create_features(data, target_col)
    X_test = full_feat.iloc[-12:][feature_cols]
    y_test = full_feat.iloc[-12:][target_col]

    preds = rf_val.predict(X_test)

    # Weighted Accuracy
    total_error = np.sum(np.abs(y_test - preds))
    total_actual = np.sum(y_test)

    if total_actual == 0: return "N/A (Zeros)"

    wmape = total_error / total_actual
    accuracy = max(0, 100 * (1 - wmape))

    return f"{accuracy:.1f}%"

# --- 2. Dashboard Logic ---
if not df.empty:
    min_year = int(df['YEAR'].min())
    max_year = int(df['YEAR'].max())

    # Widgets
    mun_widget = widgets.Dropdown(options=sorted(df['MUNICIPALITY'].unique()), value='ATOK', description='Municipality:')
    farm_widget = widgets.Dropdown(options=sorted(df['FARM TYPE'].unique()), value='IRRIGATED', description='Farm Type:')
    crop_widget = widgets.Dropdown(options=sorted(df['CROP'].unique()), value='CABBAGE', description='Crop:')

    year_range_slider = widgets.IntRangeSlider(
        value=[max_year - 2, max_year + 1],
        min=min_year, max=max_year + 5, step=1,
        description='Year Range:', continuous_update=False, layout=widgets.Layout(width='600px')
    )

    def update_plot(municipality, farm_type, crop, year_range):
        try:
            start_year, end_year = year_range

            # Filter Data
            subset = df[(df['MUNICIPALITY'] == municipality) & (df['FARM TYPE'] == farm_type) & (df['CROP'] == crop)].copy()
            if subset.empty: print(f"No data for {crop}."); return

            subset.set_index('Date', inplace=True)

            subset_resampled = subset[['Production(mt)', 'Productivity(mt/ha)', 'Rain Value (mm)']].resample('MS').mean().fillna(0)

            if len(subset_resampled) < 24: print("Not enough data to train."); return

            feature_cols = ['lag_1', 'lag_2', 'lag_12', 'Month', 'Year', 'Rain Value (mm)']

            # --- 1. CALCULATE ACCURACY ---
            acc_prod = calculate_accuracy(subset_resampled, 'Production(mt)', feature_cols)
            acc_yield = calculate_accuracy(subset_resampled, 'Productivity(mt/ha)', feature_cols)

            # --- 2. TRAIN FINAL MODEL ---
            train_prod = create_features(subset_resampled, 'Production(mt)')
            rf_prod = RandomForestRegressor(n_estimators=200, random_state=42)
            rf_prod.fit(train_prod[feature_cols], train_prod['Production(mt)'])

            train_yield = create_features(subset_resampled, 'Productivity(mt/ha)')
            rf_yield = RandomForestRegressor(n_estimators=200, random_state=42)
            rf_yield.fit(train_yield[feature_cols], train_yield['Productivity(mt/ha)'])

            # --- 3. FORECAST LOOP ---
            last_date = subset_resampled.index[-1]
            target_date = pd.Timestamp(year=end_year, month=12, day=1)

            curr_prod = list(subset_resampled['Production(mt)'])
            curr_yield = list(subset_resampled['Productivity(mt/ha)'])
            monthly_avg_rain = subset_resampled.groupby(subset_resampled.index.month)['Rain Value (mm)'].mean()

            future_dates, future_prod, future_yield = [], [], []

            if target_date > last_date:
                dates_to_predict = pd.date_range(start=last_date + pd.DateOffset(months=1), end=target_date, freq='MS')
                for date in dates_to_predict:
                    feat = {
                        'lag_1': curr_prod[-1], 'lag_2': curr_prod[-2],
                        'lag_12': curr_prod[-12] if len(curr_prod) >= 12 else np.mean(curr_prod),
                        'Month': date.month, 'Year': date.year,
                        'Rain Value (mm)': monthly_avg_rain.get(date.month, 0)
                    }
                    pred_p = rf_prod.predict(pd.DataFrame([feat])[feature_cols])[0]
                    # Add Noise (Volatility)
                    pred_p = max(0, pred_p + np.random.normal(0, np.std(curr_prod) * 0.1))

                    feat['lag_1'] = curr_yield[-1]; feat['lag_2'] = curr_yield[-2]
                    feat['lag_12'] = curr_yield[-12] if len(curr_yield) >= 12 else np.mean(curr_yield)
                    pred_y = rf_yield.predict(pd.DataFrame([feat])[feature_cols])[0]

                    curr_prod.append(pred_p); curr_yield.append(pred_y)
                    future_dates.append(date); future_prod.append(pred_p); future_yield.append(pred_y)

            # --- 4. PLOTTING ---
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 10), sharex=True)

            full_dates = subset_resampled.index.union(future_dates)
            series_p = pd.concat([subset_resampled['Production(mt)'], pd.Series(future_prod, index=future_dates)])
            series_y = pd.concat([subset_resampled['Productivity(mt/ha)'], pd.Series(future_yield, index=future_dates)])

            view_mask = (full_dates.year >= start_year) & (full_dates.year <= end_year)
            view_dates = full_dates[view_mask]
            cutoff = subset_resampled.index[-1]

            ax1.plot(series_p[series_p.index <= cutoff].reindex(view_dates), color='#1f77b4', linewidth=3, label='Historical')
            if len(future_dates) > 0:
                ax1.plot(series_p[series_p.index >= cutoff].reindex(view_dates), color='#ff7f0e', linestyle='--', linewidth=3, label='Forecast')

            ax1.set_title(f"Production Forecast: {crop} | Model Accuracy: {acc_prod}", fontsize=16, fontweight='bold')
            ax1.set_ylabel('Metric Tons'); ax1.legend(); ax1.grid(True, alpha=0.3)

            ax2.plot(series_y[series_y.index <= cutoff].reindex(view_dates), color='#2ca02c', linewidth=3, label='Historical')
            if len(future_dates) > 0:
                ax2.plot(series_y[series_y.index >= cutoff].reindex(view_dates), color='#d62728', linestyle='--', linewidth=3, label='Forecast')

            ax2.set_title(f"Yield Forecast | Model Accuracy: {acc_yield}", fontsize=14)
            ax2.set_ylabel('Yield (MT/Ha)'); ax2.legend(); ax2.grid(True, alpha=0.3)

            plt.show()

            print(f"Model Confidence (Accuracy on last 12 months): Production={acc_prod}, Yield={acc_yield}")

        except Exception: print(traceback.format_exc())

    output = widgets.interactive_output(update_plot, {'municipality': mun_widget, 'farm_type': farm_widget, 'crop': crop_widget, 'year_range': year_range_slider})
    display(widgets.VBox([widgets.HBox([mun_widget, farm_widget]), widgets.HBox([crop_widget, year_range_slider])]), output)

else: print("Error: No data loaded.")

Status: Initializing Dashboard...
Success: Loaded /content/FINAL_MASTER_DATASET.csv


Output()